# LeetCode 1389 — Create Target Array in the Given Order bằng Python

**Link đề:** https://leetcode.com/problems/create-target-array-in-the-given-order/
**Độ khó:** Easy · **Chủ đề:** Array, Simulation

```python
class Solution:
    def createTargetArray(self, nums: list[int], index: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Mô phỏng (simulation) đúng mô tả đề bằng `list.insert(i, x)` và hiểu nó làm gì **bên trong** (dịch phần tử sang phải).
2. Tự cài đặt phép chèn bằng vòng lặp dịch phần tử — thấy rõ vì sao chèn tốn `O(n)`.
3. Học cách nghĩ **ngược**: xử lý từ cuối lên, đặt mỗi phần tử vào "ô trống thứ k" — và tăng tốc bằng **Fenwick tree** (cây chỉ số nhị phân) tới `O(n log n)`.
4. Trace từng bước, chứng minh, phân tích Time/Space, test đối chiếu, tự giải biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho hai mảng `nums` và `index` cùng độ dài. Tạo mảng `target`:
- Ban đầu `target` rỗng.
- Đọc từ trái sang phải: với mỗi `i`, **chèn** `nums[i]` vào vị trí `index[i]` của `target`.
- Trả về `target`.

Đề đảm bảo mọi phép chèn đều hợp lệ (`0 <= index[i] <= i`, tức vị trí chèn không vượt quá độ dài hiện tại).

**Ví dụ 1:** `nums = [0,1,2,3,4]`, `index = [0,1,2,2,1]`

| nums | index | target |
|---|---|---|
| 0 | 0 | [0] |
| 1 | 1 | [0,1] |
| 2 | 2 | [0,1,2] |
| 3 | 2 | [0,1,3,2] |
| 4 | 1 | [0,4,1,3,2] |

**Ví dụ 2:** `nums = [1,2,3,4,0]`, `index = [0,1,2,3,0]` → `[0,1,2,3,4]`.
**Ví dụ 3:** `nums = [1]`, `index = [0]` → `[1]`.

**Constraints:** `1 <= nums.length, index.length <= 100`; `nums.length == index.length`; `0 <= nums[i] <= 100`; `0 <= index[i] <= i`.

## 2. Nền tảng: `list.insert(i, x)`

`lst.insert(i, x)` chèn `x` vào **trước** phần tử đang ở vị trí `i`; các phần tử từ `i` trở đi **dịch sang phải** một ô. Nếu `i == len(lst)` thì tương đương `append` (thêm cuối).

In [1]:
lst = [10, 20, 30]
lst.insert(1, 99)
print(lst)                 # [10, 99, 20, 30]
lst.insert(len(lst), 7)    # chèn ở cuối = append
print(lst)
lst.insert(0, -1)          # chèn đầu
print(lst)

[10, 99, 20, 30]
[10, 99, 20, 30, 7]
[-1, 10, 99, 20, 30, 7]


**Cẩn thận:** Python **không báo lỗi** khi `i` vượt độ dài — nó lặng lẽ chèn vào cuối. Điều này khác với gán `lst[i] = x` (báo `IndexError`). Bài này đảm bảo `index[i] <= i` nên không gặp vấn đề, nhưng khi tự cài đặt cần biết.

In [2]:
a = [1, 2]
a.insert(100, 3)           # vượt độ dài → vẫn chèn cuối
print(a)
try:
    a[100] = 4
except IndexError as e:
    print("Gán a[100] → IndexError:", e)

[1, 2, 3]
Gán a[100] → IndexError: list assignment index out of range


### 2.1. `zip` để đọc hai mảng song song

`zip(nums, index)` cho các cặp `(nums[i], index[i])` — gọn hơn dùng `range(len(nums))`.

In [3]:
nums1, index1 = [0, 1, 2, 3, 4], [0, 1, 2, 2, 1]
for x, pos in zip(nums1, index1):
    print(f"chèn {x} vào vị trí {pos}")

chèn 0 vào vị trí 0
chèn 1 vào vị trí 1
chèn 2 vào vị trí 2
chèn 3 vào vị trí 2
chèn 4 vào vị trí 1


## 3. Cách 1 — Mô phỏng bằng `insert` (lời giải trực tiếp)

Làm đúng như đề mô tả.

In [4]:
def create_target_insert(nums, index):
    target = []
    for x, pos in zip(nums, index):
        target.insert(pos, x)
    return target

print(create_target_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))   # [0, 4, 1, 3, 2]
print(create_target_insert([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]))   # [0, 1, 2, 3, 4]
print(create_target_insert([1], [0]))                           # [1]

[0, 4, 1, 3, 2]
[0, 1, 2, 3, 4]
[1]


## 4. TRACE — mô phỏng ví dụ 1

In [5]:
def trace_create_target(nums, index):
    target = []
    print(f"{'i':>2} | {'nums[i]':>7} | {'index[i]':>8} | target sau khi chèn")
    for i, (x, pos) in enumerate(zip(nums, index)):
        before = target[:]
        target.insert(pos, x)
        moved = before[pos:]
        note = f"(dịch {moved} sang phải)" if moved else "(chèn cuối)"
        print(f"{i:>2} | {x:>7} | {pos:>8} | {target} {note}")
    return target

trace_create_target([0, 1, 2, 3, 4], [0, 1, 2, 2, 1])

 i | nums[i] | index[i] | target sau khi chèn
 0 |       0 |        0 | [0] (chèn cuối)
 1 |       1 |        1 | [0, 1] (chèn cuối)
 2 |       2 |        2 | [0, 1, 2] (chèn cuối)
 3 |       3 |        2 | [0, 1, 3, 2] (dịch [2] sang phải)
 4 |       4 |        1 | [0, 4, 1, 3, 2] (dịch [1, 3, 2] sang phải)


[0, 4, 1, 3, 2]

Cú pháp mới:
- `for i, (x, pos) in enumerate(zip(...))` — `enumerate` cho `(i, (x, pos))`, ta **unpack lồng** ngay trong `for`.
- `target[:]` — slicing toàn bộ = **bản sao** của list (để in trạng thái trước khi chèn mà không bị thay đổi theo).

Trace cho thấy mỗi lần chèn ở giữa, **toàn bộ phần đuôi** phải dịch sang phải. Đó là chi phí ẩn của `insert`.

## 5. Cách 2 — Tự cài đặt phép chèn bằng dịch phần tử

Để hiểu `insert` tốn bao nhiêu, ta tự làm:
1. Thêm một ô ở cuối (`append` tạm giá trị bất kỳ).
2. Đi từ cuối về `pos + 1`, chép `target[k] = target[k - 1]` (dịch phải).
3. Gán `target[pos] = x`.

Số lần chép = `len(target) - pos` → chèn ở đầu tốn nhiều nhất.

In [6]:
def create_target_manual(nums, index):
    target = []
    shifts = 0
    for x, pos in zip(nums, index):
        target.append(None)                        # nới thêm một ô
        for k in range(len(target) - 1, pos, -1):  # k = cuối .. pos+1
            target[k] = target[k - 1]
            shifts += 1
        target[pos] = x
    return target, shifts

print(create_target_manual([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))
print(create_target_manual([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]))

([0, 4, 1, 3, 2], 4)
([0, 1, 2, 3, 4], 4)


Hàm trả **tuple** `(target, shifts)` để thấy số lần dịch. Ví dụ 2 dịch 4 lần ở bước cuối (chèn `0` vào đầu một list dài 4).

### Trường hợp xấu nhất: luôn chèn vào đầu
`index = [0, 0, ..., 0]` → lần thứ `i` dịch `i` phần tử → tổng `0 + 1 + ... + (n-1) = n(n-1)/2` = **`O(n²)`**.

In [7]:
for n in [10, 100, 1000]:
    _, s = create_target_manual(list(range(n)), [0] * n)
    print(f"n={n:>5}: số lần dịch = {s:>7} = n(n-1)/2 = {n * (n - 1) // 2}")

n=   10: số lần dịch =      45 = n(n-1)/2 = 45
n=  100: số lần dịch =    4950 = n(n-1)/2 = 4950
n= 1000: số lần dịch =  499500 = n(n-1)/2 = 499500


Với `n <= 100` của đề, `O(n²)` = vài nghìn phép — rất nhanh. Nhưng nếu `n = 10^5` thì `~5·10^9` phép là quá chậm. Phần tiếp theo trình bày cách nghĩ giúp đạt `O(n log n)` — kiến thức nâng cao, hữu ích cho bài khó cùng dạng (ví dụ *Queue Reconstruction by Height*, LeetCode 406).

## 6. Insight — nghĩ **ngược từ cuối**

Phần tử **cuối cùng** được chèn (`i = n-1`) **không bị ai đẩy** nữa → vị trí cuối cùng của nó trong `target` chính là `index[n-1]`.

Phần tử `i = n-2`: sau nó chỉ có phần tử `n-1` chèn vào. Nếu bỏ qua ô mà phần tử `n-1` chiếm, thì phần tử `n-2` nằm ở vị trí `index[n-2]` **trong các ô còn lại**.

Tổng quát: xử lý `i` từ `n-1` về `0`; phần tử `i` được đặt vào **ô trống thứ `index[i]`** (đếm từ 0, theo thứ tự trái → phải) của mảng kết quả dài `n`.

Lý do (trực giác): tại thời điểm chèn, `nums[i]` có đúng `index[i]` phần tử đứng trước nó; các phần tử chèn sau có thể chen vào trước nó, nhưng chúng đều đã được đặt rồi (chiếm ô). Vậy trong các ô **chưa** bị phần tử sau chiếm, `nums[i]` đứng ở vị trí `index[i]`.

In [8]:
def create_target_reverse_naive(nums, index):
    n = len(nums)
    target = [None] * n
    for i in range(n - 1, -1, -1):
        k = index[i]                   # cần ô trống thứ k (đếm từ 0)
        for pos in range(n):
            if target[pos] is None:
                if k == 0:
                    target[pos] = nums[i]
                    break
                k -= 1
    return target

print(create_target_reverse_naive([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))
print(create_target_reverse_naive([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]))

[0, 4, 1, 3, 2]
[0, 1, 2, 3, 4]


### 6.1. TRACE cách nghĩ ngược trên ví dụ 1

`n = 5`, xử lý `i = 4, 3, 2, 1, 0`:

| i | nums[i] | index[i] | Ô trống trước bước | Đặt vào ô | target sau bước |
|---|---|---|---|---|---|
| 4 | 4 | 1 | 0,1,2,3,4 | ô trống thứ 1 = **1** | [_, 4, _, _, _] |
| 3 | 3 | 2 | 0,2,3,4 | ô trống thứ 2 = **3** | [_, 4, _, 3, _] |
| 2 | 2 | 2 | 0,2,4 | ô trống thứ 2 = **4** | [_, 4, _, 3, 2] |
| 1 | 1 | 1 | 0,2 | ô trống thứ 1 = **2** | [_, 4, 1, 3, 2] |
| 0 | 0 | 0 | 0 | ô trống thứ 0 = **0** | [0, 4, 1, 3, 2] |

Khớp đáp án `[0, 4, 1, 3, 2]`.

In [9]:
def trace_reverse(nums, index):
    n = len(nums)
    target = ["_"] * n
    for i in range(n - 1, -1, -1):
        free = [p for p in range(n) if target[p] == "_"]
        slot = free[index[i]]
        target[slot] = nums[i]
        print(f"i={i}: nums={nums[i]}, index={index[i]} | ô trống {free} → đặt vào ô {slot} | {target}")
    return target

trace_reverse([0, 1, 2, 3, 4], [0, 1, 2, 2, 1])

i=4: nums=4, index=1 | ô trống [0, 1, 2, 3, 4] → đặt vào ô 1 | ['_', 4, '_', '_', '_']
i=3: nums=3, index=2 | ô trống [0, 2, 3, 4] → đặt vào ô 3 | ['_', 4, '_', 3, '_']
i=2: nums=2, index=2 | ô trống [0, 2, 4] → đặt vào ô 4 | ['_', 4, '_', 3, 2]
i=1: nums=1, index=1 | ô trống [0, 2] → đặt vào ô 2 | ['_', 4, 1, 3, 2]
i=0: nums=0, index=0 | ô trống [0] → đặt vào ô 0 | [0, 4, 1, 3, 2]


[0, 4, 1, 3, 2]

Bản ngây thơ này vẫn `O(n²)` (mỗi bước quét tìm ô trống). Điểm lợi: bài toán đã trở thành **"tìm ô trống thứ k"** — một truy vấn có cấu trúc dữ liệu chuyên dụng.

## 7. Cách 3 (nâng cao) — Fenwick tree tìm "ô trống thứ k" trong `O(log n)`

### 7.1. Fenwick tree (Binary Indexed Tree) là gì?

Cấu trúc lưu một mảng `a[1..n]` và hỗ trợ trong `O(log n)`:
- `update(i, delta)`: `a[i] += delta`.
- `prefix(i)`: tổng `a[1] + ... + a[i]`.
- `find_kth(k)`: vị trí nhỏ nhất `p` mà `prefix(p) >= k` (khi mọi `a[i] >= 0`).

Ý tưởng: `tree[i]` lưu tổng của một **khối** kết thúc tại `i`, độ dài khối = **bit 1 thấp nhất** của `i`, tính bằng `i & (-i)`.

In [10]:
for i in range(1, 9):
    print(f"i={i} ({i:04b}): i & -i = {i & -i} → tree[{i}] phụ trách a[{i - (i & -i) + 1}..{i}]")

i=1 (0001): i & -i = 1 → tree[1] phụ trách a[1..1]
i=2 (0010): i & -i = 2 → tree[2] phụ trách a[1..2]
i=3 (0011): i & -i = 1 → tree[3] phụ trách a[3..3]
i=4 (0100): i & -i = 4 → tree[4] phụ trách a[1..4]
i=5 (0101): i & -i = 1 → tree[5] phụ trách a[5..5]
i=6 (0110): i & -i = 2 → tree[6] phụ trách a[5..6]
i=7 (0111): i & -i = 1 → tree[7] phụ trách a[7..7]
i=8 (1000): i & -i = 8 → tree[8] phụ trách a[1..8]


In [11]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)          # dùng chỉ số 1..n

    def update(self, i, delta):
        while i <= self.n:
            self.tree[i] += delta
            i += i & -i                    # nhảy lên khối cha

    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & -i                    # bỏ khối hiện tại
        return s

    def find_kth(self, k):
        # vị trí nhỏ nhất p (1-based) sao cho prefix(p) >= k
        pos = 0
        step = 1 << (self.n.bit_length())
        while step:
            nxt = pos + step
            if nxt <= self.n and self.tree[nxt] < k:
                pos = nxt
                k -= self.tree[nxt]
            step >>= 1
        return pos + 1

fw = Fenwick(5)
for i in range(1, 6):
    fw.update(i, 1)                        # 5 ô, đều trống (giá trị 1)
print("prefix:", [fw.prefix(i) for i in range(1, 6)])
print("ô trống thứ 1, 2, 5:", fw.find_kth(1), fw.find_kth(2), fw.find_kth(5))
fw.update(2, -1)                           # ô 2 bị chiếm
print("sau khi chiếm ô 2 → ô trống thứ 2 là:", fw.find_kth(2))

prefix: [1, 2, 3, 4, 5]
ô trống thứ 1, 2, 5: 1 2 5
sau khi chiếm ô 2 → ô trống thứ 2 là: 3


Giải thích `find_kth`: ta "leo" từ trên xuống theo các lũy thừa của 2 (`step = 2^t`, rồi `2^(t-1)`, …). Nếu khối `tree[pos + step]` còn **chưa đủ** `k`, ta nhảy qua nó và trừ bớt `k`. Cuối cùng `pos + 1` là vị trí cần tìm. `n.bit_length()` = số bit của `n` (ví dụ `5 = 101₂` → 3).

### 7.2. Ghép lại
- Đánh dấu mọi ô `1..n` là trống (`a[p] = 1`).
- Với `i` từ `n-1` về `0`: `p = find_kth(index[i] + 1)` (ô trống thứ `index[i]` đếm từ 0 = thứ `index[i]+1` đếm từ 1), đặt `target[p-1] = nums[i]`, rồi `update(p, -1)`.

In [12]:
def create_target_fenwick(nums, index):
    n = len(nums)
    fw = Fenwick(n)
    for p in range(1, n + 1):
        fw.update(p, 1)
    target = [0] * n
    for i in range(n - 1, -1, -1):
        p = fw.find_kth(index[i] + 1)
        target[p - 1] = nums[i]
        fw.update(p, -1)
    return target

print(create_target_fenwick([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))
print(create_target_fenwick([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]))
print(create_target_fenwick([1], [0]))

[0, 4, 1, 3, 2]
[0, 1, 2, 3, 4]
[1]


## 8. Code nộp LeetCode

Với `n <= 100`, lời giải `insert` là chuẩn mực — ngắn, rõ, đúng.

In [13]:
class Solution:
    def createTargetArray(self, nums: list[int], index: list[int]) -> list[int]:
        target = []
        for x, pos in zip(nums, index):
            target.insert(pos, x)
        return target

sol = Solution()
print(sol.createTargetArray([0,1,2,3,4], [0,1,2,2,1]))   # [0,4,1,3,2]
print(sol.createTargetArray([1,2,3,4,0], [0,1,2,3,0]))   # [0,1,2,3,4]
print(sol.createTargetArray([1], [0]))                   # [1]

[0, 4, 1, 3, 2]
[0, 1, 2, 3, 4]
[1]


Bản Fenwick (cũng Accepted, dùng khi `n` lớn):

In [14]:
class SolutionFenwick:
    def createTargetArray(self, nums: list[int], index: list[int]) -> list[int]:
        n = len(nums)
        tree = [0] * (n + 1)
        for p in range(1, n + 1):            # khởi tạo O(n): tree[p] = độ dài khối = p & -p
            tree[p] = p & -p
        LOG = 1 << n.bit_length()
        target = [0] * n
        for i in range(n - 1, -1, -1):
            k, pos, step = index[i] + 1, 0, LOG
            while step:
                nxt = pos + step
                if nxt <= n and tree[nxt] < k:
                    pos, k = nxt, k - tree[nxt]
                step >>= 1
            target[pos] = nums[i]            # ô 1-based pos+1 ↔ chỉ số 0-based pos
            j = pos + 1
            while j <= n:
                tree[j] -= 1
                j += j & -j
        return target

print(SolutionFenwick().createTargetArray([0,1,2,3,4], [0,1,2,2,1]))

[0, 4, 1, 3, 2]


Mẹo khởi tạo: khi mọi `a[p] = 1`, tổng khối của `tree[p]` = **độ dài khối** = `p & -p`, nên không cần gọi `update` `n` lần.

## 9. Vì sao đúng?

**Cách 1 (insert):** làm đúng định nghĩa của đề — mỗi bước chèn `nums[i]` vào vị trí `index[i]`; `insert` của Python thực hiện chính xác phép đó (dịch phần đuôi). Đề đảm bảo `index[i] <= i = len(target)` nên mọi phép chèn hợp lệ.

**Cách nghĩ ngược — invariant:** sau khi xử lý các `i' > i` (theo thứ tự giảm), các ô đã điền chứa đúng các phần tử `nums[i']` tại **vị trí cuối cùng** của chúng. Xét `nums[i]`: lúc vừa chèn, có đúng `index[i]` phần tử đứng trước nó (đều là phần tử chèn trước `i`). Các phần tử chèn **trước** `i` có thứ tự tương đối với `nums[i]` không bao giờ đổi về sau (chèn thêm chỉ thêm phần tử mới vào giữa). Những phần tử chen vào **trước** `nums[i]` về sau đều là các `i' > i` — đã chiếm ô. Vậy trong các ô **chưa chiếm**, `nums[i]` đứng sau đúng `index[i]` ô → nó nằm ở **ô trống thứ `index[i]`** (đếm từ 0). Khi `i = 0` xong, mọi ô đã đầy và đúng vị trí.

## 10. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. `list.insert` | `O(n²)` worst | `O(n)` (kết quả) | `insert` dịch phần đuôi, nhưng được cài bằng C nên rất nhanh |
| 2. Tự dịch phần tử | `O(n²)` | `O(n)` | minh hoạ chi phí |
| Ngược, quét ô trống | `O(n²)` | `O(n)` | đổi bài thành "ô trống thứ k" |
| 3. Ngược + Fenwick | `O(n log n)` | `O(n)` | nâng cao |

In [15]:
import time, random
rng = random.Random(0)
n = 20000
nums_big = [rng.randint(0, 100) for _ in range(n)]
index_big = [rng.randint(0, i) for i in range(n)]
for name, f in [("insert", create_target_insert), ("fenwick", create_target_fenwick)]:
    t0 = time.perf_counter()
    res = f(nums_big, index_big)
    print(f"{name:>8}: {time.perf_counter() - t0:.3f}s")
assert create_target_insert(nums_big, index_big) == create_target_fenwick(nums_big, index_big)
print("Hai cách cho cùng kết quả với n =", n)

  insert: 0.020s
 fenwick: 0.060s


Hai cách cho cùng kết quả với n = 20000


Thời gian cụ thể phụ thuộc máy; điều quan trọng là **bậc tăng**: `insert` là `O(n²)` (dù hằng số rất nhỏ vì dịch bộ nhớ bằng C), Fenwick là `O(n log n)` (nhưng mỗi bước chạy bằng Python nên hằng số lớn hơn). Với `n` vừa phải, `insert` thậm chí có thể nhanh hơn — "Big-O tốt hơn" không luôn có nghĩa "nhanh hơn ở mọi kích thước".

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.createTargetArray([0,1,2,3,4], [0,1,2,2,1]), [0,4,1,3,2])
check("đề 2", sol.createTargetArray([1,2,3,4,0], [0,1,2,3,0]), [0,1,2,3,4])
check("đề 3", sol.createTargetArray([1], [0]), [1])
check("luôn chèn đầu", sol.createTargetArray([1, 2, 3], [0, 0, 0]), [3, 2, 1])
check("luôn chèn cuối", sol.createTargetArray([1, 2, 3], [0, 1, 2]), [1, 2, 3])
check("giá trị trùng", sol.createTargetArray([5, 5, 7], [0, 0, 1]), [5, 7, 5])

PASS đề 1: [0, 4, 1, 3, 2]
PASS đề 2: [0, 1, 2, 3, 4]
PASS đề 3: [1]
PASS luôn chèn đầu: [3, 2, 1]
PASS luôn chèn cuối: [1, 2, 3]
PASS giá trị trùng: [5, 7, 5]


In [17]:
methods = [create_target_insert, lambda a, b: create_target_manual(a, b)[0], create_target_reverse_naive,
           create_target_fenwick, Solution().createTargetArray, SolutionFenwick().createTargetArray]
rng = random.Random(1389)
for _ in range(500):
    n = rng.randint(1, 40)
    nums = [rng.randint(0, 100) for _ in range(n)]
    index = [rng.randint(0, i) for i in range(n)]       # đảm bảo 0 <= index[i] <= i
    expected = create_target_insert(nums, index)
    for f in methods:
        assert f(list(nums), list(index)) == expected
    assert sorted(expected) == sorted(nums)             # kết quả là hoán vị của nums
print(f"500 bộ ngẫu nhiên × {len(methods)} cách: khớp nhau — PASS")

500 bộ ngẫu nhiên × 6 cách: khớp nhau — PASS


`lambda a, b: create_target_manual(a, b)[0]` — bọc hàm trả tuple thành hàm chỉ trả `target`, để dùng chung vòng test.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Gán thay vì chèn
`target[index[i]] = nums[i]` **ghi đè** phần tử cũ thay vì đẩy nó sang phải (và lỗi chỉ số khi list còn ngắn).

In [18]:
def wrong_assign(nums, index):
    target = [None] * len(nums)
    for x, pos in zip(nums, index):
        target[pos] = x                  # ghi đè!
    return target

print("Sai :", wrong_assign([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]), "| Đúng:", create_target_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))

Sai : [0, 4, 3, None, None] | Đúng: [0, 4, 1, 3, 2]


### Lỗi 2: Đảo thứ tự tham số `insert(x, pos)`
Chữ ký là `insert(vị_trí, giá_trị)`. Đảo lại vẫn chạy (vì cả hai là số) nhưng sai kết quả — loại lỗi khó phát hiện.

In [19]:
def wrong_arg_order(nums, index):
    target = []
    for x, pos in zip(nums, index):
        target.insert(x, pos)           # sai thứ tự
    return target

print("Sai :", wrong_arg_order([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]), "| Đúng:", create_target_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))

Sai : [0, 1, 2, 2, 1] | Đúng: [0, 4, 1, 3, 2]


### Lỗi 3: Dịch phần tử theo chiều **xuôi**
Khi tự dịch, nếu đi từ `pos` lên cuối (`target[k+1] = target[k]`) thì giá trị `target[pos]` bị chép **lan** ra mọi ô sau. Phải dịch **từ cuối về**.

In [20]:
def wrong_shift_forward(nums, index):
    target = []
    for x, pos in zip(nums, index):
        target.append(None)
        for k in range(pos, len(target) - 1):      # đi xuôi — SAI
            target[k + 1] = target[k]
        target[pos] = x
    return target

print("Sai :", wrong_shift_forward([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]), "| Đúng:", create_target_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))

Sai : [0, 4, 1, 1, 1] | Đúng: [0, 4, 1, 3, 2]


### Lỗi 4: Cách nghĩ ngược nhưng xử lý **xuôi**
Đặt phần tử vào "ô trống thứ k" theo thứ tự `i = 0, 1, ...` là sai: phần tử đầu tiên vẫn còn bị các phần tử sau đẩy đi.

In [21]:
def wrong_forward_slots(nums, index):
    n = len(nums)
    target = [None] * n
    for i in range(n):                              # xuôi — SAI
        free = [p for p in range(n) if target[p] is None]
        target[free[index[i]]] = nums[i]
    return target

try:
    print("Sai :", wrong_forward_slots([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]))
except IndexError as e:
    print("Sai : IndexError —", e, "(ô trống thứ k không còn tồn tại)")
print("Sai :", wrong_forward_slots([1, 2, 3], [0, 0, 0]), "| Đúng:", create_target_insert([1, 2, 3], [0, 0, 0]))

Sai : IndexError — list index out of range (ô trống thứ k không còn tồn tại)
Sai : [1, 2, 3] | Đúng: [3, 2, 1]


### Mẹo nhận dạng pattern

- Đề mô tả **từng bước thao tác** trên mảng với `n` nhỏ → **mô phỏng trực tiếp** là lời giải đúng đắn nhất.
- "Chèn vào vị trí k" lặp nhiều lần với `n` lớn → nghĩ **ngược từ cuối**: phần tử chèn sau cùng có vị trí cố định → "ô trống thứ k" → **Fenwick tree / segment tree**.
- Bài anh em: *Queue Reconstruction by Height* (406), *Count of Smaller Numbers After Self* (315), *Design Front Middle Back Queue* (1670), *Shuffle the Array* (1470).

## 13. Biến thể

### 13.1. Xoá theo chỉ số
Ngược lại với chèn: cho list và dãy vị trí, lần lượt **xoá** phần tử tại vị trí đó (`list.pop(i)`), trả về thứ tự các phần tử bị xoá.

In [22]:
def removal_order(arr, positions):
    a = arr[:]
    removed = []
    for p in positions:
        removed.append(a.pop(p))
    return removed

print(removal_order([10, 20, 30, 40], [1, 0, 1, 0]))   # [20, 10, 40, 30]

[20, 10, 40, 30]


`list.pop(i)` xoá và **trả về** phần tử ở vị trí `i` (mặc định phần tử cuối), các phần tử sau dịch trái — cũng `O(n)`.

### 13.2. Queue Reconstruction by Height (LeetCode 406)
Mỗi người `[h, k]`: cao `h`, có đúng `k` người **cao hơn hoặc bằng** đứng trước. Sắp theo `h` giảm dần (hòa thì `k` tăng), rồi **chèn** mỗi người vào vị trí `k` — đúng kỹ thuật của bài này!

In [23]:
def reconstruct_queue(people):
    people = sorted(people, key=lambda p: (-p[0], p[1]))
    queue = []
    for h, k in people:
        queue.insert(k, [h, k])
    return queue

print(reconstruct_queue([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]))
# [[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]


Vì sao chèn được? Người cao hơn đã đứng sẵn; người thấp hơn chèn sau **không ảnh hưởng** điều kiện của người cao (họ không đếm người thấp hơn). Nên chèn người đang xét vào vị trí `k` là đủ.

### 13.3. Tìm ngược `index` từ `target`
Cho `nums` (thứ tự chèn) và `target` (kết quả), hỏi có tồn tại `index` hợp lệ không, và tìm nó. (Xem bài tập 6.)

## 14. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
`nums = [4, 2, 4, 3, 2]`, `index = [0, 0, 1, 3, 1]`. Lập bảng trạng thái `target` sau mỗi bước.

### Bài 2 — Viết lại bằng `insert`

In [24]:
def exercise_insert(nums, index):
    # TODO
    pass

# Gợi ý test:
# assert exercise_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]) == [0, 4, 1, 3, 2]

### Bài 3 — Tự cài `my_insert(lst, pos, x)` không dùng `insert`

In [25]:
def exercise_my_insert(lst, pos, x):
    # TODO: append rồi dịch từ cuối về pos+1, cuối cùng gán lst[pos] = x
    pass

# Gợi ý test:
# a = [1, 2, 3]; exercise_my_insert(a, 1, 9); assert a == [1, 9, 2, 3]
# a = []; exercise_my_insert(a, 0, 5); assert a == [5]

### Bài 4 — Cách nghĩ ngược (quét ô trống)

In [26]:
def exercise_reverse(nums, index):
    # TODO: i từ n-1 về 0, đặt nums[i] vào ô trống thứ index[i]
    pass

# Gợi ý test:
# assert exercise_reverse([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]) == [0, 1, 2, 3, 4]

### Bài 5 — Vị trí cuối cùng của mỗi phần tử
Trả về list `final_pos` với `final_pos[i]` = chỉ số của `nums[i]` (phần tử chèn ở bước `i`) trong `target` cuối cùng. Lưu ý `nums` có thể trùng giá trị nên đừng tìm theo giá trị.

In [27]:
def exercise_final_positions(index):
    # TODO
    pass

# Gợi ý test:
# assert exercise_final_positions([0, 1, 2, 2, 1]) == [0, 2, 4, 3, 1]

### Bài 6 — Mini challenge: khôi phục `index`
Cho `target` (các giá trị **phân biệt**) và thứ tự chèn `nums` (một hoán vị của `target`), tìm `index` sao cho `createTargetArray(nums, index) == target`.

In [28]:
def exercise_recover_index(nums, target):
    # TODO
    pass

# Gợi ý test:
# assert exercise_recover_index([0, 1, 2, 3, 4], [0, 4, 1, 3, 2]) == [0, 1, 2, 2, 1]

## 15. Gợi ý 3 mức

**Bài 1** — (1) Làm từng bước `insert`. (2) Bước 2 chèn `2` vào đầu. (3) Kết quả cuối `[2, 2, 4, 4, 3]`.

**Bài 2** — (1) `zip(nums, index)`. (2) `target.insert(pos, x)`. (3) Trả `target`.

**Bài 3** — (1) Nới list một ô. (2) `for k in range(len(lst) - 1, pos, -1): lst[k] = lst[k-1]`. (3) `lst[pos] = x`.

**Bài 4** — (1) Mục 6. (2) `free = [p for p in range(n) if target[p] is None]`. (3) `target[free[index[i]]] = nums[i]`.

**Bài 5** — (1) Mô phỏng nhưng chèn **số thứ tự bước** `i` thay vì giá trị. (2) `order.insert(index[i], i)`. (3) Sau cùng `final_pos[order[p]] = p`.

**Bài 6** — (1) Khi chèn `nums[i]`, `index[i]` = số phần tử **đã chèn** (bước `< i`) đứng **trước** nó trong `target`. (2) `pos = {v: p for p, v in enumerate(target)}`. (3) `index[i] = sum(1 for j in range(i) if pos[nums[j]] < pos[nums[i]])`.

## 16. Đáp án tham khảo

**Bài 1:** `[4]` → `[2,4]` → `[2,4,4]` → `[2,4,4,3]` → `[2,2,4,4,3]`.

In [29]:
assert trace_create_target([4, 2, 4, 3, 2], [0, 0, 1, 3, 1]) == [2, 2, 4, 4, 3]

def answer_insert(nums, index):
    target = []
    for x, pos in zip(nums, index):
        target.insert(pos, x)
    return target

def answer_my_insert(lst, pos, x):
    lst.append(None)
    for k in range(len(lst) - 1, pos, -1):
        lst[k] = lst[k - 1]
    lst[pos] = x

def answer_reverse(nums, index):
    n = len(nums)
    target = [None] * n
    for i in range(n - 1, -1, -1):
        free = [p for p in range(n) if target[p] is None]
        target[free[index[i]]] = nums[i]
    return target

def answer_final_positions(index):
    order = []
    for i, pos in enumerate(index):
        order.insert(pos, i)
    final_pos = [0] * len(index)
    for p, i in enumerate(order):
        final_pos[i] = p
    return final_pos

def answer_recover_index(nums, target):
    pos = {v: p for p, v in enumerate(target)}
    return [sum(1 for j in range(i) if pos[nums[j]] < pos[nums[i]]) for i in range(len(nums))]

assert answer_insert([0, 1, 2, 3, 4], [0, 1, 2, 2, 1]) == [0, 4, 1, 3, 2]
a = [1, 2, 3]; answer_my_insert(a, 1, 9); assert a == [1, 9, 2, 3]
a = []; answer_my_insert(a, 0, 5); assert a == [5]
assert answer_reverse([1, 2, 3, 4, 0], [0, 1, 2, 3, 0]) == [0, 1, 2, 3, 4]
assert answer_final_positions([0, 1, 2, 2, 1]) == [0, 2, 4, 3, 1]
assert answer_recover_index([0, 1, 2, 3, 4], [0, 4, 1, 3, 2]) == [0, 1, 2, 2, 1]

rng = random.Random(21)
for _ in range(300):
    n = rng.randint(1, 25)
    index = [rng.randint(0, i) for i in range(n)]
    nums = rng.sample(range(1000), n)                    # phân biệt cho bài 6
    tgt = answer_insert(nums, index)
    assert answer_reverse(nums, index) == tgt
    fp = answer_final_positions(index)
    assert all(tgt[fp[i]] == nums[i] for i in range(n))
    assert answer_recover_index(nums, tgt) == index
print("Đáp án bài tập: PASS toàn bộ")

 i | nums[i] | index[i] | target sau khi chèn
 0 |       4 |        0 | [4] (chèn cuối)
 1 |       2 |        0 | [2, 4] (dịch [4] sang phải)
 2 |       4 |        1 | [2, 4, 4] (dịch [4] sang phải)
 3 |       3 |        3 | [2, 4, 4, 3] (chèn cuối)
 4 |       2 |        1 | [2, 2, 4, 4, 3] (dịch [4, 4, 3] sang phải)
Đáp án bài tập: PASS toàn bộ


**Nhận xét bài 6:** `index` khôi phục được **duy nhất** — vì `index[i]` buộc phải bằng số phần tử chèn trước đó đứng trước `nums[i]` (thứ tự tương đối giữa các phần tử đã có không bao giờ thay đổi). Assert cuối kiểm chứng điều này trên 300 bộ ngẫu nhiên.

## 17. Tổng kết — template nhớ nhanh + checklist

```python
target = []
for x, pos in zip(nums, index):
    target.insert(pos, x)          # insert(VỊ_TRÍ, GIÁ_TRỊ), đuôi dịch phải
return target
```

Ý tưởng nâng cao cần nhớ: **xử lý ngược từ cuối** → phần tử thứ `i` nằm ở **ô trống thứ `index[i]`** → Fenwick tree `find_kth` cho `O(n log n)`.

**Checklist trước khi submit**
- [ ] Dùng **chèn** (`insert`), không gán đè.
- [ ] Đúng thứ tự tham số `insert(pos, x)`.
- [ ] Nếu tự dịch phần tử: dịch **từ cuối về**.
- [ ] Nếu nghĩ ngược: duyệt `i` **giảm dần**.
- [ ] Đã test chèn toàn ở đầu, toàn ở cuối, có giá trị trùng.